# 16S and metabolomics CTF Analyses of the Longitudinal Acne Study

Date last updated: 12/30/25

Notebook author: Britta De Pessemier, Yang Chen 

Data analysis by: Tyler Myers, Britta De Pessemier, and Yang Chen

In [56]:
# Import essential Python packages
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Ellipse
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D

# Scientific and statistical packages
import scipy.stats as ss
from skbio import DistanceMatrix
from skbio.stats.distance import permanova

# BIOM and Qiime2 related packages
import biom
import qiime2 as q2
from biom import load_table
from qiime2 import Artifact
import h5py

# Gemelli packages
from qiime2 import Artifact, Metadata
from qiime2.plugins import gemelli
from skbio.stats.ordination import OrdinationResults

# Stats
from skbio.stats.distance import DistanceMatrix
from skbio.stats.distance import permanova
from scipy.spatial.distance import pdist, squareform
from statsmodels.stats.multitest import multipletests
import itertools
from itertools import combinations

# Other
import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message="You passed a edgecolor/edgecolors .*"
)


## Import table and metadata

In [57]:
biom_paths = {
    "16S_V1-V3": "../Data/16S/Tables/179426_feature-table_16S_V1V3_rare-11054_sampleIDfixed.biom",
    "16S_V4": "../Data/16S/Tables/174951_feature-table_16S_V4_rare-3769_sampleIDfixed.biom",
    "metabolomics": "..//Data/metabolomics/Run3_10252024/output/metabolomics_table_final_sampleIDfixed.biom"
}

metadata_path = "../Metadata/metadata_final_22102024.tsv"
metadata_df = pd.read_csv(metadata_path, sep="\t")
metadata_df = metadata_df.set_index("SampleID")
metadata_df.index.name = '#SampleID'

In [58]:
# Make qiime2 compatible metadata
qiime_md = Metadata(metadata_df)

## Plotting preparation

In [59]:
def draw_ellipse(mean, cov, ax, color, scale_factor=1.5):
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
    width, height = 2 * np.sqrt(eigenvalues) * scale_factor

    ell = Ellipse(
        xy=mean,
        width=width,
        height=height,
        angle=angle,
        edgecolor=color,
        facecolor=color,
        alpha=0.2,
        linewidth=0.5
    )
    ax.add_patch(ell)

In [60]:
group_colors = {
    "Healthy": "#000096",
    "Acne_L": "#ff676c",
    "Acne_NL": "#17c6d5"
}

group_name_mapping = {
    "Healthy": "Healthy",
    "Acne_L": "Acne Lesional",
    "Acne_NL": "Acne Non-lesional"
}

In [61]:
def build_tensor_robust(table, md, individual_id, state_column):
    """
    Build gemelli CTF tensor across gemelli versions.
    """

    b = build()

    if hasattr(b, "__call__"):
        return b(table, md, individual_id, state_column)

    if hasattr(b, "fit_transform"):
        return b.fit_transform(
            table=table,
            sample_metadata=md,
            individual_id_column=individual_id,
            state_column=state_column
        )

    if hasattr(b, "build"):
        return b.build(
            table,
            md,
            individual_id_column=individual_id,
            state_column=state_column
        )

    raise RuntimeError("Unsupported gemelli build API")


In [62]:
def calculate_pairwise_permanova(ordination_df, group_column='Group', n_permutations=999):
    """
    Calculate pairwise PERMANOVA between groups
    """
    # Create distance matrix from ordination coordinates
    coords = ordination_df[['PC1', 'PC2', 'PC3']].values
    dist_array = squareform(pdist(coords, metric='euclidean'))
    
    # Create DistanceMatrix object with sample IDs
    dm = DistanceMatrix(dist_array, ids=ordination_df.index.tolist())
    
    # Get unique groups
    groups = ordination_df[group_column].unique()
    group_pairs = list(combinations(groups, 2))
    
    # Calculate pairwise PERMANOVA
    pairwise_results = {}
    
    for group1, group2 in group_pairs:
        # Get sample IDs for the two groups
        ids1 = ordination_df[ordination_df[group_column] == group1].index.tolist()
        ids2 = ordination_df[ordination_df[group_column] == group2].index.tolist()
        pair_ids = ids1 + ids2
        
        # Filter distance matrix for these samples
        pair_dm = dm.filter(pair_ids)
        
        # Create grouping vector
        pair_grouping = [group1] * len(ids1) + [group2] * len(ids2)
        
        # Run PERMANOVA
        result = permanova(pair_dm, pair_grouping, permutations=n_permutations)
        
        pairwise_results[f"{group1}_vs_{group2}"] = {
            'statistic': result['test statistic'],
            'p_value': result['p-value'],
            'R2': result['test statistic'] / (result['number of groups'] - 1)  # pseudo R²
        }
    
    return pairwise_results

## Run Compositional Tensor Factorization (CTF) (C1 - left cheek)

In [63]:
ctf_results = {}

for region, biom_path in biom_paths.items():

    print(f"\nRunning CTF for {region}")

    # Load BIOM
    table_artifact = Artifact.import_data(
        "FeatureTable[Frequency]",
        biom_path
    )
    table_df = table_artifact.view(pd.DataFrame)

    # Match samples between BIOM and metadata
    shared_samples = table_df.index.intersection(metadata_df.index)
    if shared_samples.empty:
        raise ValueError(f"No shared samples for {region}")

    table_df = table_df.loc[shared_samples]
    md_all = metadata_df.loc[shared_samples].copy()

    # REMOVE C2 SAMPLES HERE
    keep_idx = md_all["c_zone"] != "C2"

    table_df = table_df.loc[keep_idx]
    md_all = md_all.loc[keep_idx]

    print(f"  Samples after removing C2: {table_df.shape[0]}")

    if table_df.empty:
        raise ValueError(f"All samples removed after filtering C2 for {region}")

    # Re-import filtered table
    table_art = Artifact.import_data(
        "FeatureTable[Frequency]",
        table_df
    )

    # QIIME metadata
    md_all.index.name = "#SampleID"
    qiime_md = Metadata(md_all)

    # RUN CTF (C2 removed)
    ctf_res = gemelli.actions.ctf(
        table=table_art,
        sample_metadata=qiime_md,
        individual_id_column="subject_ID",
        state_column="day",
        n_components=3
    )

    # Subject biplot
    ordination = ctf_res.subject_biplot.view(OrdinationResults)

    spca_df = ordination.samples.iloc[:, :3].copy()
    spca_df.columns = ["PC1", "PC2", "PC3"]

    # Subject-level metadata
    subj_md = (
        md_all
        .drop_duplicates("subject_ID")
        .set_index("subject_ID")
    )

    spca_df["Group"] = spca_df.index.map(subj_md["group"])
    spca_df["day"] = spca_df.index.map(subj_md["day"])
    spca_df["c_zone"] = spca_df.index.map(subj_md["c_zone"])

    # Store results
    ctf_results[region] = {
        "ordination": ordination,
        "spca_df": spca_df,
        "metadata": subj_md
    }

    feature_df = (
    ordination.features
    .iloc[:, :3]
    .copy()
)

    # Extract feature loadings
    feature_df.columns = ["PC1", "PC2", "PC3"]
    feature_df["feature_id"] = feature_df.index

    # Sort by strongest overall loading (max abs across PCs)
    feature_df = (
        feature_df
        .assign(
            max_abs_loading=lambda x: x[["PC1", "PC2", "PC3"]].abs().max(axis=1)
        )
        .sort_values("max_abs_loading", ascending=False)
        .drop(columns="max_abs_loading")
    )

    # Save
    outdir = "../Analyses/16S/CTF/feature_loadings"
    os.makedirs(outdir, exist_ok=True)

    feature_df.to_csv(
        f"{outdir}/CTF_feature_loadings_{region}_left-cheek.tsv",
        sep="\t",
        index=False
    )

    # Save the subject biplot as qza
    qza_outdir = "../Analyses/16S/CTF/subject_biplot_qza"
    os.makedirs(qza_outdir, exist_ok=True)

    subject_biplot_qza = f"{qza_outdir}/CTF_subject_biplot_{region}_left-cheek.qza"

    ctf_res.subject_biplot.save(subject_biplot_qza)

    print(f"  Saved subject biplot QZA: {subject_biplot_qza}")



Running CTF for 16S_V1-V3
  Samples after removing C2: 128


/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/gemelli/preprocessing.py:968: RuntimeWarning: Subject(s) (PP_304,PP_306,PP_314,PP_313,PP_305,PP_310,PP_319,PP_308,PP_318,PP_317) contains multiple samples. Multiple subject counts will be meaned across samples by subject.
  warnings.warn(''.join(["Subject(s) (", str(duplicated_ids),


  Saved subject biplot QZA: ../Analyses/16S/CTF/subject_biplot_qza/CTF_subject_biplot_16S_V1-V3_left-cheek.qza

Running CTF for 16S_V4
  Samples after removing C2: 129


/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/gemelli/preprocessing.py:968: RuntimeWarning: Subject(s) (PP_304,PP_306,PP_314,PP_313,PP_305,PP_310,PP_319,PP_308,PP_318,PP_317) contains multiple samples. Multiple subject counts will be meaned across samples by subject.
  warnings.warn(''.join(["Subject(s) (", str(duplicated_ids),


  Saved subject biplot QZA: ../Analyses/16S/CTF/subject_biplot_qza/CTF_subject_biplot_16S_V4_left-cheek.qza

Running CTF for metabolomics
  Samples after removing C2: 158


/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/gemelli/preprocessing.py:968: RuntimeWarning: Subject(s) (PP_304,PP_306,PP_314,PP_313,PP_305,PP_310,PP_319,PP_308,PP_318,PP_317) contains multiple samples. Multiple subject counts will be meaned across samples by subject.
  warnings.warn(''.join(["Subject(s) (", str(duplicated_ids),


  Saved subject biplot QZA: ../Analyses/16S/CTF/subject_biplot_qza/CTF_subject_biplot_metabolomics_left-cheek.qza


In [64]:
# Set seed for reproducibility
np.random.seed(69)

for region, res in ctf_results.items():

    ordination = res["ordination"]
    spca_df = res["spca_df"]

    # Calculate PERMANOVA (will use the seed set above)
    permanova_results = calculate_pairwise_permanova(spca_df, group_column='Group')
    
    # Extract p-values for adjustment
    comparisons = list(permanova_results.keys())
    p_values = [results['p_value'] for results in permanova_results.values()]
    
    # Apply Benjamini-Hochberg FDR adjustment
    _, adjusted_p_values, _, _ = multipletests(p_values, method='fdr_bh')
    
    # Update results with adjusted p-values
    for i, comparison in enumerate(comparisons):
        permanova_results[comparison]['adjusted_p_value'] = adjusted_p_values[i]

    pc1_var = ordination.proportion_explained[0] * 100
    pc2_var = ordination.proportion_explained[1] * 100

    fig, ax = plt.subplots(figsize=(8, 8))  # Increased width for annotations

    plot_df = spca_df.copy()

    # Create group name mapping for legend
    group_abbrev = {
        'Healthy': 'H',
        'Acne_L': 'AL',
        'Acne_NL': 'ANL'
    }

    for group, df_g in plot_df.groupby("Group"):

        color = group_colors[group]
        
        # Use abbreviated name for legend
        label_name = group_abbrev.get(group, group)

        ax.scatter(
            df_g["PC1"],
            df_g["PC2"],
            s=120,
            color=color,
            edgecolor="black",
            linewidth=0.6,
            alpha=1,
            label=label_name  # Use abbreviated name
        )

        if len(df_g) > 2:
            pts = df_g[["PC1", "PC2"]].values
            draw_ellipse(
                pts.mean(axis=0),
                np.cov(pts, rowvar=False),
                ax,
                color
            )

    ax.set_xlabel(f"PC1 ({pc1_var:.2f}%)", fontsize=16)
    ax.set_ylabel(f"PC2 ({pc2_var:.2f}%)", fontsize=16)
    ax.set_title(f"CTF {region} (left cheek)", fontsize=18)

    # Customize legend with frame and manual location
    ax.legend(
        frameon=True,  # Enable frame
        fancybox=True,  # Rounded corners
        edgecolor='grey',  # Grey frame color
        framealpha=1.0,  # Fully opaque frame
        bbox_to_anchor=(0.96, 0.86)
    )
    
    # If you need the legend with grey frame and 1.0 thickness exactly
    legend = ax.get_legend()
    legend.get_frame().set_linewidth(1.0)
    
    # Add PERMANOVA annotations with adjusted p-values
    text_y = 0.925
    text_x = 0.62
    
    # Collect all annotation text
    annotation_texts = []
    for comparison, results in permanova_results.items():
        # Split and map group names
        group1, group2 = comparison.split('_vs_')
        group1_abbrev = group_abbrev.get(group1, group1)
        group2_abbrev = group_abbrev.get(group2, group2)
        groups = f"{group1_abbrev} vs {group2_abbrev}"
        
        # Use adjusted p-value for significance symbols
        adj_p = results['adjusted_p_value']
        raw_p = results['p_value']
        
        # Format p-value with adjusted significance
        if adj_p < 0.001:
            p_text = f"*** p={adj_p:.3f}"
        elif adj_p < 0.01:
            p_text = f"** p={adj_p:.3f}"
        elif adj_p < 0.05:
            p_text = f"* p={adj_p:.3f}"
        else:
            p_text = f"p={adj_p:.3f}"
        
        # Create annotation text
        text = f"{groups}: {p_text}, F={results['statistic']:.3f}"
        annotation_texts.append(text)

    # Join all annotations with newlines
    full_annotation = "PERMANOVA (FDR-adjusted):\n" + "\n".join(annotation_texts)

    # Add text box with frame
    props = dict(
        boxstyle='round,pad=0.5',  # Round corners with padding
        facecolor='white',          # White background
        edgecolor='grey',          # Grey border
        linewidth=1,              # Border width
        alpha=0.9                   # Slight transparency
    )

    ax.text(
        text_x, 
        text_y + 0.05,  # Adjust starting position
        full_annotation, 
        transform=ax.transAxes,
        fontsize=11,
        verticalalignment='top',
        bbox=props
    )
    
    plt.tight_layout()

    plt.savefig(
        f"../Figures/Main/Figure_2/CTF_{region}_left-cheek.png",
        dpi=600,
        bbox_inches='tight'
    )
    plt.close()
    
    # Print results to console as well
    print(f"\n=== PERMANOVA results for {region} ===")
    for comparison, results in permanova_results.items():
        print(f"{comparison}:")
        print(f"  Raw p-value: {results['p_value']:.3f}")
        print(f"  FDR-adjusted q-value: {results['adjusted_p_value']:.3f}")
        print(f"  F-statistic: {results['statistic']:.3f}")


=== PERMANOVA results for 16S_V1-V3 ===
Healthy_vs_Acne_NL:
  Raw p-value: 0.071
  FDR-adjusted q-value: 0.106
  F-statistic: 2.333
Healthy_vs_Acne_L:
  Raw p-value: 0.005
  FDR-adjusted q-value: 0.015
  F-statistic: 4.491
Acne_NL_vs_Acne_L:
  Raw p-value: 0.592
  FDR-adjusted q-value: 0.592
  F-statistic: 0.725

=== PERMANOVA results for 16S_V4 ===
Healthy_vs_Acne_NL:
  Raw p-value: 0.024
  FDR-adjusted q-value: 0.072
  F-statistic: 3.435
Healthy_vs_Acne_L:
  Raw p-value: 0.070
  FDR-adjusted q-value: 0.105
  F-statistic: 2.373
Acne_NL_vs_Acne_L:
  Raw p-value: 0.488
  FDR-adjusted q-value: 0.488
  F-statistic: 0.756

=== PERMANOVA results for metabolomics ===
Healthy_vs_Acne_L:
  Raw p-value: 0.001
  FDR-adjusted q-value: 0.003
  F-statistic: 5.764
Healthy_vs_Acne_NL:
  Raw p-value: 0.073
  FDR-adjusted q-value: 0.110
  F-statistic: 2.429
Acne_L_vs_Acne_NL:
  Raw p-value: 0.504
  FDR-adjusted q-value: 0.504
  F-statistic: 0.713


## Run Compositional Tensor Factorization (CTF) (C2 - right cheek)

In [65]:
ctf_results = {}

for region, biom_path in biom_paths.items():

    print(f"\nRunning CTF for {region}")

    # Load BIOM
    table_artifact = Artifact.import_data(
        "FeatureTable[Frequency]",
        biom_path
    )
    table_df = table_artifact.view(pd.DataFrame)

    # Match samples between BIOM and metadata
    shared_samples = table_df.index.intersection(metadata_df.index)
    if shared_samples.empty:
        raise ValueError(f"No shared samples for {region}")

    table_df = table_df.loc[shared_samples]
    md_all = metadata_df.loc[shared_samples].copy()

    # REMOVE C1 SAMPLES HERE
    keep_idx = md_all["c_zone"] != "C1"

    table_df = table_df.loc[keep_idx]
    md_all = md_all.loc[keep_idx]

    print(f"  Samples after removing C1: {table_df.shape[0]}")

    if table_df.empty:
        raise ValueError(f"All samples removed after filtering C2 for {region}")

    # Re-import filtered table
    table_art = Artifact.import_data(
        "FeatureTable[Frequency]",
        table_df
    )

    # QIIME metadata
    md_all.index.name = "#SampleID"
    qiime_md = Metadata(md_all)

    # RUN CTF (C1 removed)
    ctf_res = gemelli.actions.ctf(
        table=table_art,
        sample_metadata=qiime_md,
        individual_id_column="subject_ID",
        state_column="day",
        n_components=3
    )

    # Subject biplot
    ordination = ctf_res.subject_biplot.view(OrdinationResults)

    spca_df = ordination.samples.iloc[:, :3].copy()
    spca_df.columns = ["PC1", "PC2", "PC3"]

    # Subject-level metadata
    subj_md = (
        md_all
        .drop_duplicates("subject_ID")
        .set_index("subject_ID")
    )

    spca_df["Group"] = spca_df.index.map(subj_md["group"])
    spca_df["day"] = spca_df.index.map(subj_md["day"])
    spca_df["c_zone"] = spca_df.index.map(subj_md["c_zone"])

    # Store results
    ctf_results[region] = {
        "ordination": ordination,
        "spca_df": spca_df,
        "metadata": subj_md
    }

    # Extract feature loadings
    feature_df = (
        ordination.features
        .iloc[:, :3]
        .copy()
    )
    feature_df.columns = ["PC1", "PC2", "PC3"]
    feature_df["feature_id"] = feature_df.index

    # Sort by strongest overall loading (max abs across PCs)
    feature_df = (
        feature_df
        .assign(
            max_abs_loading=lambda x: x[["PC1", "PC2", "PC3"]].abs().max(axis=1)
        )
        .sort_values("max_abs_loading", ascending=False)
        .drop(columns="max_abs_loading")
    )

    # Save
    outdir = "../Analyses/16S/CTF/feature_loadings"
    os.makedirs(outdir, exist_ok=True)

    feature_df.to_csv(
        f"{outdir}/CTF_feature_loadings_{region}_right-cheek.tsv",
        sep="\t",
        index=False
    )

    # Save the subject biplot as qza
    qza_outdir = "../Analyses/16S/CTF/subject_biplot_qza"
    os.makedirs(qza_outdir, exist_ok=True)

    subject_biplot_qza = f"{qza_outdir}/CTF_subject_biplot_{region}_right-cheek.qza"

    ctf_res.subject_biplot.save(subject_biplot_qza)

    print(f"  Saved subject biplot QZA: {subject_biplot_qza}")



Running CTF for 16S_V1-V3
  Samples after removing C1: 128


/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/gemelli/preprocessing.py:968: RuntimeWarning: Subject(s) (PP_304,PP_306,PP_314,PP_313,PP_305,PP_310,PP_319,PP_308,PP_318,PP_317) contains multiple samples. Multiple subject counts will be meaned across samples by subject.
  warnings.warn(''.join(["Subject(s) (", str(duplicated_ids),


  Saved subject biplot QZA: ../Analyses/16S/CTF/subject_biplot_qza/CTF_subject_biplot_16S_V1-V3_right-cheek.qza

Running CTF for 16S_V4
  Samples after removing C1: 133


/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/gemelli/preprocessing.py:968: RuntimeWarning: Subject(s) (PP_304,PP_306,PP_314,PP_313,PP_305,PP_310,PP_319,PP_308,PP_318,PP_317) contains multiple samples. Multiple subject counts will be meaned across samples by subject.
  warnings.warn(''.join(["Subject(s) (", str(duplicated_ids),


  Saved subject biplot QZA: ../Analyses/16S/CTF/subject_biplot_qza/CTF_subject_biplot_16S_V4_right-cheek.qza

Running CTF for metabolomics
  Samples after removing C1: 158


/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/gemelli/preprocessing.py:968: RuntimeWarning: Subject(s) (PP_304,PP_306,PP_314,PP_313,PP_305,PP_310,PP_319,PP_308,PP_318,PP_317) contains multiple samples. Multiple subject counts will be meaned across samples by subject.
  warnings.warn(''.join(["Subject(s) (", str(duplicated_ids),


  Saved subject biplot QZA: ../Analyses/16S/CTF/subject_biplot_qza/CTF_subject_biplot_metabolomics_right-cheek.qza


In [66]:
# Set seed for reproducibility
np.random.seed(69)

for region, res in ctf_results.items():

    ordination = res["ordination"]
    spca_df = res["spca_df"]

    # Calculate PERMANOVA (will use the seed set above)
    permanova_results = calculate_pairwise_permanova(spca_df, group_column='Group')
    
    # Extract p-values for adjustment
    comparisons = list(permanova_results.keys())
    p_values = [results['p_value'] for results in permanova_results.values()]
    
    # Apply Benjamini-Hochberg FDR adjustment
    _, adjusted_p_values, _, _ = multipletests(p_values, method='fdr_bh')
    
    # Update results with adjusted p-values
    for i, comparison in enumerate(comparisons):
        permanova_results[comparison]['adjusted_p_value'] = adjusted_p_values[i]

    pc1_var = ordination.proportion_explained[0] * 100
    pc2_var = ordination.proportion_explained[1] * 100

    fig, ax = plt.subplots(figsize=(8, 8))  # Increased width for annotations

    plot_df = spca_df.copy()

    # Create group name mapping for legend
    group_abbrev = {
        'Healthy': 'H',
        'Acne_L': 'AL',
        'Acne_NL': 'ANL'
    }

    for group, df_g in plot_df.groupby("Group"):

        color = group_colors[group]
        
        # Use abbreviated name for legend
        label_name = group_abbrev.get(group, group)

        ax.scatter(
            df_g["PC1"],
            df_g["PC2"],
            s=120,
            color=color,
            edgecolor="black",
            linewidth=0.6,
            alpha=1,
            label=label_name  # Use abbreviated name
        )

        if len(df_g) > 2:
            pts = df_g[["PC1", "PC2"]].values
            draw_ellipse(
                pts.mean(axis=0),
                np.cov(pts, rowvar=False),
                ax,
                color
            )

    ax.set_xlabel(f"PC1 ({pc1_var:.2f}%)", fontsize=16)
    ax.set_ylabel(f"PC2 ({pc2_var:.2f}%)", fontsize=16)
    ax.set_title(f"CTF {region} (right cheek)", fontsize=18)

    # Customize legend with frame and manual location
    ax.legend(
        frameon=True,  # Enable frame
        fancybox=True,  # Rounded corners
        edgecolor='grey',  # Grey frame color
        framealpha=1.0,  # Fully opaque frame
        bbox_to_anchor=(0.96, 0.86)
    )
    
    # If you need the legend with grey frame and 1.0 thickness exactly
    legend = ax.get_legend()
    legend.get_frame().set_linewidth(1.0)
    
    # Add PERMANOVA annotations with adjusted p-values
    text_y = 0.925
    text_x = 0.62
    
    # Collect all annotation text
    annotation_texts = []
    for comparison, results in permanova_results.items():
        # Split and map group names
        group1, group2 = comparison.split('_vs_')
        group1_abbrev = group_abbrev.get(group1, group1)
        group2_abbrev = group_abbrev.get(group2, group2)
        groups = f"{group1_abbrev} vs {group2_abbrev}"
        
        # Use adjusted p-value for significance symbols
        adj_p = results['adjusted_p_value']
        raw_p = results['p_value']
        
        # Format p-value with adjusted significance
        if adj_p < 0.001:
            p_text = f"*** p={adj_p:.3f}"
        elif adj_p < 0.01:
            p_text = f"** p={adj_p:.3f}"
        elif adj_p < 0.05:
            p_text = f"* p={adj_p:.3f}"
        else:
            p_text = f"p={adj_p:.3f}"
        
        # Create annotation text
        text = f"{groups}: {p_text}, F={results['statistic']:.3f}"
        annotation_texts.append(text)

    # Join all annotations with newlines
    full_annotation = "PERMANOVA (FDR-adjusted):\n" + "\n".join(annotation_texts)

    # Add text box with frame
    props = dict(
        boxstyle='round,pad=0.5',  # Round corners with padding
        facecolor='white',          # White background
        edgecolor='grey',          # Grey border
        linewidth=1,              # Border width
        alpha=0.9                   # Slight transparency
    )

    ax.text(
        text_x, 
        text_y + 0.05,  # Adjust starting position
        full_annotation, 
        transform=ax.transAxes,
        fontsize=11,
        verticalalignment='top',
        bbox=props
    )
    
    plt.tight_layout()

    plt.savefig(
        f"../Figures/Main/Figure_2/CTF_{region}_right-cheek.png",
        dpi=600,
        bbox_inches='tight'
    )
    plt.close()
    
    # Print results to console as well
    print(f"\n=== PERMANOVA results for {region} ===")
    for comparison, results in permanova_results.items():
        print(f"{comparison}:")
        print(f"  Raw p-value: {results['p_value']:.3f}")
        print(f"  FDR-adjusted q-value: {results['adjusted_p_value']:.3f}")
        print(f"  F-statistic: {results['statistic']:.3f}")


=== PERMANOVA results for 16S_V1-V3 ===
Healthy_vs_Acne_NL:
  Raw p-value: 0.001
  FDR-adjusted q-value: 0.003
  F-statistic: 4.316
Healthy_vs_Acne_L:
  Raw p-value: 0.028
  FDR-adjusted q-value: 0.042
  F-statistic: 2.868
Acne_NL_vs_Acne_L:
  Raw p-value: 0.120
  FDR-adjusted q-value: 0.120
  F-statistic: 2.238

=== PERMANOVA results for 16S_V4 ===
Healthy_vs_Acne_L:
  Raw p-value: 0.006
  FDR-adjusted q-value: 0.018
  F-statistic: 3.964
Healthy_vs_Acne_NL:
  Raw p-value: 0.019
  FDR-adjusted q-value: 0.029
  F-statistic: 3.215
Acne_L_vs_Acne_NL:
  Raw p-value: 0.479
  FDR-adjusted q-value: 0.479
  F-statistic: 0.933

=== PERMANOVA results for metabolomics ===
Healthy_vs_Acne_NL:
  Raw p-value: 0.136
  FDR-adjusted q-value: 0.204
  F-statistic: 2.043
Healthy_vs_Acne_L:
  Raw p-value: 0.001
  FDR-adjusted q-value: 0.003
  F-statistic: 8.927
Acne_NL_vs_Acne_L:
  Raw p-value: 0.449
  FDR-adjusted q-value: 0.449
  F-statistic: 1.055


## Map taxonomy of each ASV feature loading within each CTF model

In [67]:
# Paths
ctf_dir = "../Analyses/16S/CTF/feature_loadings"
taxonomy_path = "../Taxonomy/174116_taxonomy.tsv"

# Load SILVA taxonomy
tax_df = pd.read_csv(taxonomy_path, sep="\t")

# Build ASV -> taxonomy mapping
asv_to_tax = (
    tax_df
    .set_index("Feature ID")["Taxon"]
    .to_dict()
)

# Find all CTF TSVs EXCEPT metabolomics
ctf_files = [
    f for f in glob.glob(os.path.join(ctf_dir, "*.tsv"))
    if "metabolomics" not in os.path.basename(f).lower()
]

print(f"Found {len(ctf_files)} 16S CTF files (metabolomics excluded)")

# Annotate each file
for f in ctf_files:
    print(f"Annotating {os.path.basename(f)}")

    df = pd.read_csv(f, sep="\t")

    if "feature_id" not in df.columns:
        raise ValueError(f"'feature_id' column missing in {f}")

    # Map taxonomy
    df["taxonomy"] = df["feature_id"].map(asv_to_tax)

    # Explicitly label missing taxonomy
    df["taxonomy"] = df["taxonomy"].fillna("Unassigned")

    # Save new file
    out_path = f.replace(".tsv", "_with_taxonomy.tsv")
    df.to_csv(out_path, sep="\t", index=False)


Found 4 16S CTF files (metabolomics excluded)
Annotating CTF_feature_loadings_16S_V4_right-cheek.tsv
Annotating CTF_feature_loadings_16S_V4_left-cheek.tsv
Annotating CTF_feature_loadings_16S_V1-V3_left-cheek.tsv
Annotating CTF_feature_loadings_16S_V1-V3_right-cheek.tsv


In [68]:
# Cleanup
ctf_dir = "../Analyses/16S/CTF/feature_loadings"

old_files = [
    f for f in glob.glob(os.path.join(ctf_dir, "*.tsv"))
    if (
        not f.endswith("_with_taxonomy.tsv")      # old (unannotated)
        and "metabolomics" not in os.path.basename(f).lower()  # not metabolomics
    )
]

print(f"Deleting {len(old_files)} old 16S TSV files")

for f in old_files:
    print(f"Deleting {os.path.basename(f)}")
    os.remove(f)

Deleting 4 old 16S TSV files
Deleting CTF_feature_loadings_16S_V4_right-cheek.tsv
Deleting CTF_feature_loadings_16S_V4_left-cheek.tsv
Deleting CTF_feature_loadings_16S_V1-V3_left-cheek.tsv
Deleting CTF_feature_loadings_16S_V1-V3_right-cheek.tsv
